# Books Sales and Ratings

I chose this dataset after considering a few options, including movies and wellbeing. Books have a mix of ratings, genres, prices and sales figures that can be compared.

**Main question:** how do book ratings, genres, prices and publishers relate to the sales figures in this dataset?

**Source:** [Books Sales and Ratings on Kaggle](https://www.kaggle.com/datasets/thedevastator/books-sales-and-ratings), shared by thedevastator. The description credits [Josh Murrey](https://data.world/josh-nbu). The CSV was downloaded on 21 October 2026.

The analysis uses Pandas for the data checks and calculations, and Matplotlib for all six charts. The code is followed by Markdown explaining the steps and results. The findings describe this dataset, rather than the whole book market.

## 1. Loading the data

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from IPython.display import display

# Find the project folder from either the root or notebooks folder.
ROOT = Path.cwd()
if not (ROOT / "data/raw/manifest.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "data/raw/manifest.json").exists():
    raise FileNotFoundError("Open this notebook from the project folder or notebooks folder.")

DATA_PATH = ROOT / "data/raw/Books_Data_Clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run python scripts/download_data.py from the project folder first.")

manifest = json.loads((ROOT / "data/raw/manifest.json").read_text())
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == manifest["sha256"], "Check the dataset version."
raw = pd.read_csv(DATA_PATH)

FIGURES = ROOT / "reports/figures"
TABLES = ROOT / "reports/tables"
PROCESSED = ROOT / "data/processed"
for folder in [FIGURES, TABLES, PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]}")
display(raw.head(3))

Rows: 1,070 | Columns: 15


,index,Publishing Year,Book Name,Author,language_code,Author_Rating,Book_average_rating,Book_ratings_count,genre,gross sales,publisher revenue,sale price,sales rank,Publisher,units sold
0,0,1975.0,Beowulf,"Unknown, Seamus Heaney",en-US,Novice,3.42,155903,genre fiction,34160.0,20496.0,4.88,1,HarperCollins Publishers,7000
1,1,1987.0,Batman: Year One,"Frank Miller, David Mazzucchelli, Richmond Lew...",eng,Intermediate,4.23,145267,genre fiction,12437.5,7462.5,1.99,2,HarperCollins Publishers,6250
2,2,2015.0,Go Set a Watchman,Harper Lee,eng,Novice,3.31,138669,genre fiction,47795.0,28677.0,8.69,3,"Amazon Digital Services, Inc.",5500


`read_csv()` loads the original file into a DataFrame called `raw`. The path checks let the notebook run from the main folder or the notebooks folder. The checksum checks that the CSV is the same version recorded in the project.

The file has **1,070 rows and 15 columns**. Each row is a book record. The first rows give an idea of the values, but they aren't enough to judge the whole dataset. The folders above hold the cleaned copy, summary tables and saved charts.

## 2. Initial checks

In [2]:
audit = pd.DataFrame({
    "column": [repr(name) for name in raw.columns],
    "type": raw.dtypes.astype(str).to_numpy(),
    "missing": raw.isna().sum().to_numpy(),
    "missing_percent": raw.isna().mean().mul(100).round(2).to_numpy(),
    "unique_values": raw.nunique().to_numpy(),
})
display(audit)
raw.info()
display(raw.describe(include="all").T)
print("Exact duplicates:", raw.duplicated().sum())
print("Duplicates without the index column:", raw.drop(columns="index").duplicated().sum())
audit.to_csv(TABLES / "initial_data_checks.csv", index=False)

,column,type,missing,missing_percent,unique_values
0,'index',int64,0,0.00,1070
1,'Publishing Year',float64,1,0.09,150
2,'Book Name',str,23,2.15,1045
3,'Author',str,0,0.00,735
4,'language_code',str,53,4.95,8
5,'Author_Rating',str,0,0.00,4
6,'Book_average_rating',float64,0,0.00,134
7,'Book_ratings_count',int64,0,0.00,1064
8,'genre',str,0,0.00,4
9,'gross sales',float64,0,0.00,831


<class 'pandas.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                1070 non-null   int64  
 1   Publishing Year      1069 non-null   float64
 2   Book Name            1047 non-null   str    
 3   Author               1070 non-null   str    
 4   language_code        1017 non-null   str    
 5   Author_Rating        1070 non-null   str    
 6   Book_average_rating  1070 non-null   float64
 7   Book_ratings_count   1070 non-null   int64  
 8   genre                1070 non-null   str    
 9   gross sales          1070 non-null   float64
 10  publisher revenue    1070 non-null   float64
 11  sale price           1070 non-null   float64
 12  sales rank           1070 non-null   int64  
 13  Publisher            1070 non-null   str    
 14  units sold           1070 non-null   int64  
dtypes: float64(5), int64(4), str(6)
memory usage: 125

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,1070.0,NaN,NaN,NaN,534.5,309.026698,0.0,267.25,534.5,801.75,1069.0
Publishing Year,1069.0,NaN,NaN,NaN,1971.377923,185.080257,-560.0,1985.0,2003.0,2010.0,2016.0
Book Name,1047,1045,The Awakening,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Author,1070,735,Stephen King,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language_code,1017,8,eng,737,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Author_Rating,1070,4,Intermediate,625,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Book_average_rating,1070.0,NaN,NaN,NaN,4.007,0.247244,2.97,3.85,4.015,4.17,4.77
Book_ratings_count,1070.0,NaN,NaN,NaN,94909.913084,31513.242518,27308.0,70398.0,89309.0,113906.5,206792.0
genre,1070,4,genre fiction,822,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gross sales,1070.0,NaN,NaN,NaN,1856.622944,3936.92424,104.94,372.465,809.745,1487.9575,47795.0


Exact duplicates: 0
Duplicates without the index column: 0


`info()` shows column types and non-missing counts. `describe()` gives summary statistics; `.T` puts each original column on its own row. Some statistics don't apply to text columns, so the summary shows `NaN` for those entries.

The audit found **23 missing book names**, **53 missing language codes** and **1 missing publishing year**. There are **no duplicate rows**, including when the exported index is left out. `Publisher ` has a space at the end of its name.

The year column includes values as early as **−560**. Several titles also appear to have encoding problems. Neither issue should be fixed by guessing.

## 3. Cleaning

I kept all usable records rather than dropping a whole row just because its title or language was missing. Those fields aren't needed for the five main comparisons. The original CSV stays unchanged.

In [3]:
books = raw.copy()
books.columns = books.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
books = books.rename(columns={"index": "source_row"})

# Tidy spaces while keeping names and source categories.
text_columns = ["book_name", "author", "language_code", "author_rating", "genre", "publisher"]
space_changes = {}
for column in text_columns:
    before = books[column].astype("string")
    after = before.str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA)
    space_changes[column] = int(before.fillna("<missing>").ne(after.fillna("<missing>")).sum())
    books[column] = after

display(pd.Series(space_changes, name="rows_with_spacing_changes").to_frame())

,rows_with_spacing_changes
book_name,33
author,19
language_code,0
author_rating,0
genre,0
publisher,600


Column names now use lowercase words separated by underscores, so `gross sales` becomes `gross_sales`. The exported `index` is renamed `source_row` to keep a link to the original row; it isn't used as a measure.

Extra spaces were removed from **33 book names**, **19 author entries** and **600 publisher entries**. For example, the double space in Amazon's name became a single space. This changes formatting, rather than trying to correct titles or merge publishers.

In [4]:
numeric_columns = ["source_row", "publishing_year", "book_average_rating", "book_ratings_count",
                   "gross_sales", "publisher_revenue", "sale_price", "sales_rank", "units_sold"]
conversion_losses = {}
for column in numeric_columns:
    before = books[column].notna().sum()
    books[column] = pd.to_numeric(books[column], errors="coerce")
    conversion_losses[column] = int(before - books[column].notna().sum())

# Keep whole-number fields as nullable integers.
integer_columns = ["source_row", "publishing_year", "book_ratings_count", "sales_rank", "units_sold"]
for column in integer_columns:
    values = books[column].dropna()
    assert values.mod(1).eq(0).all(), f"Review non-integer values in {column}."
    books[column] = books[column].astype("Int64")

range_checks = pd.Series({
    "ratings outside 0 to 5": int((books.book_average_rating.notna() & ~books.book_average_rating.between(0, 5)).sum()),
    "negative sales or prices": int(books[["gross_sales", "publisher_revenue", "sale_price", "units_sold"]].lt(0).any(axis=1).sum()),
    "negative rating counts": int(books.book_ratings_count.lt(0).sum()),
    "non-positive sales ranks": int(books.sales_rank.le(0).sum()),
    "future publishing years": int(books.publishing_year.gt(2026).sum()),
})
assert range_checks.eq(0).all(), "Review invalid values before continuing."
assert books.source_row.is_unique, "Review the source row identifiers."

content_columns = [column for column in books.columns if column != "source_row"]
duplicates = int(books.duplicated(subset=content_columns).sum())
books = books.drop_duplicates(subset=content_columns).copy()
print("New missing values from numeric conversion:", sum(conversion_losses.values()))
print("Duplicate rows removed after tidying:", duplicates)
display(range_checks.to_frame("rows"))

New missing values from numeric conversion: 0
Duplicate rows removed after tidying: 0


,rows
ratings outside 0 to 5,0
negative sales or prices,0
negative rating counts,0
non-positive sales ranks,0
future publishing years,0


`to_numeric()` checks that numerical fields can be read as numbers. It created **no new missing values**. Whole-number fields use Pandas' nullable `Int64` type, which allows the missing publishing year to stay missing.

The range checks found **no invalid ratings, negative sales/prices, negative rating counts, non-positive sales ranks or future publishing years**. There were still **no duplicates** after tidying spaces, so **0 rows were removed**.

The year is kept as a number rather than converted to a date. Negative years could refer to ancient works, and publishing year isn't the year a sale happened.

In [5]:
# Keep original language codes and add a readable group.
language_map = {"eng": "English", "en-US": "English", "en-GB": "English", "en-CA": "English",
                "fre": "French", "spa": "Spanish", "ara": "Arabic", "nl": "Dutch"}
books["language_group"] = books.language_code.map(language_map).fillna("Unknown")
books["publishing_year_bce"] = books.publishing_year.lt(0).fillna(False)

# This is a review flag, not an automatic correction of titles.
books["possible_encoding_issue"] = books.book_name.str.contains(r"Ã|Â|Î|Ï|â€|á¼|�", na=False)

expected_gross = books.sale_price * books.units_sold.astype(float)
books["revenue_gap"] = books.gross_sales - expected_gross
books["sales_values_match"] = np.isclose(books.gross_sales, expected_gross, atol=0.01, rtol=0)

cleaning_counts = pd.Series({
    "rows before cleaning": len(raw),
    "rows after cleaning": len(books),
    "missing book names retained": int(books.book_name.isna().sum()),
    "unknown language groups": int(books.language_group.eq("Unknown").sum()),
    "missing publishing years retained": int(books.publishing_year.isna().sum()),
    "negative publishing years flagged": int(books.publishing_year_bce.sum()),
    "possible title encoding issues flagged": int(books.possible_encoding_issue.sum()),
    "sales values that match": int(books.sales_values_match.sum()),
    "sales values that do not match": int((~books.sales_values_match).sum()),
    "zero publisher revenues retained": int(books.publisher_revenue.eq(0).sum()),
})
display(cleaning_counts.to_frame("count"))
display(books.dtypes.astype(str).rename("cleaned_type").to_frame())

books.to_csv(PROCESSED / "books_cleaned.csv", index=False)
cleaning_counts.to_csv(TABLES / "cleaning_counts.csv", header=["count"], index_label="check")
print("Saved the cleaned copy to data/processed/books_cleaned.csv")

,count
rows before cleaning,1070
rows after cleaning,1070
missing book names retained,23
unknown language groups,53
missing publishing years retained,1
negative publishing years flagged,6
possible title encoding issues flagged,20
sales values that match,444
sales values that do not match,626
zero publisher revenues retained,372


,cleaned_type
source_row,Int64
publishing_year,Int64
book_name,string
author,string
language_code,string
author_rating,string
book_average_rating,float64
book_ratings_count,Int64
genre,string
gross_sales,float64


Saved the cleaned copy to data/processed/books_cleaned.csv


All **1,070 rows** remain. The cleaned table has **20 columns**, including five additional fields for grouping and quality checks.

- Missing book names, years and original language codes are kept as missing. Only the new `language_group` uses **Unknown** for the **53** missing language codes.
- **6 negative years** are flagged rather than deleted.
- A simple text pattern flags **20 possible title encoding issues**. This is a review aid, not a complete encoding check, and the titles aren't guessed or rewritten.
- `genre fiction` and `fiction` remain separate because the source does not clearly explain their distinction. Similar publisher names are also kept separate.
- `Author_Rating` contains labels such as Famous and Novice. It is treated as a category, despite the source description calling it numeric.
- **372 zero publisher revenues** are kept. A zero may be meaningful or may reflect a recording problem; there isn't enough evidence to replace it.

`np.isclose()` checks whether gross sales equal price × units sold within **0.01**, allowing for small rounding differences. Only **444 records match**; **626 don't**. This check identifies an inconsistency, but doesn't prove which field is wrong. Sales periods, discounts or definitions could differ. None of these numbers were overwritten.

The cleaned CSV is saved locally. The public repo contains the notebook, summary tables and figures, with a script to download the same raw CSV.